# Scikit-learn classification pack — breast cancer

**Kya chahiye:** Google Colab ya local Python 3.10+. `Run all` karo; first cell missing packages install karta hai.

**Dataset:** scikit-learn ka canonical `load_breast_cancer` (569 rows, 30 features, binary diagnosis). Koi external CSV download nahi.

**Rule:** split pehle, preprocessing pipeline ke andar, test set sirf final evaluation ke liye.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "sklearn": "scikit-learn==1.5.2",
    "xgboost": "xgboost==2.1.3",
    "pandas": "pandas==2.2.3",
}
missing = [pip_name for module, pip_name in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import sklearn
import xgboost
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)
print("Runtime is ready. This pack is for Colab or a local environment, not the offline Code Bro kernel.")

## 1. Load canonical data

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame.rename(columns={"target": "diagnosis"})
print("shape:", df.shape)
print("classes:", dict(enumerate(data.target_names)))
print("balance:", df["diagnosis"].value_counts().to_dict())
df.head()

## 2. Stratified train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="diagnosis")
y = df["diagnosis"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("train:", X_train.shape, "| test:", X_test.shape)
print("train positives:", int(y_train.sum()), "| test positives:", int(y_test.sum()))

## 3. Two pipelines — same split, same metric contract

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "logistic": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42),
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "fitted")

## 4. Accuracy is not enough; inspect both classes

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

for name, model in models.items():
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    print(f"\n{name}")
    print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))
    print("confusion matrix:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, target_names=data.target_names, digits=3))

## 5. Cross-validation — one split is a sample, not a guarantee

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"{name}: ROC-AUC {scores.mean():.3f} ± {scores.std():.3f} | folds={np.round(scores, 3).tolist()}")

## 6. Permutation importance on the held-out set

In [ ]:
from sklearn.inspection import permutation_importance

best = models["random_forest"]
result = permutation_importance(best, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
order = np.argsort(result.importances_mean)[::-1][:8]
for index in order:
    print(f"{X.columns[index]:>28}: {result.importances_mean[index]: .3f} ± {result.importances_std[index]:.3f}")

## 7. What to report and what not to claim

- ROC-AUC measures ranking across thresholds; the clinical action still needs a threshold and a cost discussion.
- Permutation importance is model- and data-dependent. Correlated features can share or hide importance.
- This is a random split on one curated dataset. Before deployment, validate by site, time period and prevalence.
- `malignant` is class 0 in scikit-learn's target encoding; always print the mapping instead of assuming it.

**Exercise:** choose a threshold that gives at least 0.95 recall for the malignant class (class 0), then report the false-positive count and the expected review workload.